In [1]:
# =============================================================================
# **NOTEBOOK 2: FEATURE ENGINEERING & EDA**
# **US Accidents Severity Classification - Big Data Assignment**
# **Input:** code/data/samples/accidents_sampled.parquet
# **Output:** code/data/samples/accidents_features.parquet
# =============================================================================

import os
import sys
import time
import json
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for server environments
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType
from pyspark import StorageLevel

# MLlib pipeline components
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, MinMaxScaler, Imputer, PCA,
    Bucketizer
)
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml import Transformer

# Logging setup
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("FeatureEngineering")
# Mask home directory in all log output so no username appears
class _HomeFilter(logging.Filter):
    _h = os.path.expanduser("~")
    def filter(self, r):
        try:
            r.msg = r.getMessage().replace(self._h, "~")
            r.args = None
        except Exception:
            pass
        return True
logging.getLogger().addFilter(_HomeFilter())


# Paths
import pathlib as _pl
_cwd = _pl.Path(os.getcwd()).resolve()
BASE_DIR = (str(_cwd) if (_cwd / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent) if (_cwd.parent.parent / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent))
PARQUET_SAMPLED_PATH = os.path.join(BASE_DIR, "code", "data", "samples", "accidents_sampled.parquet")
PARQUET_FEATURES_PATH = os.path.join(BASE_DIR, "code", "data", "samples", "accidents_features.parquet")
PLOTS_DIR = os.path.join(BASE_DIR, "code", "data", "samples", "eda_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Imports and paths configured successfully.")


Imports and paths configured successfully.


In [2]:
# =============================================================================
# **SECTION 1: INITIALIZE SPARK SESSION (reuse if exists)**
# =============================================================================

spark = (
    SparkSession.builder
    .appName("USAccidents_FeatureEngineering")
    .master("local[*]")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Load full Parquet dataset (no sampling — full 1.6 GB / 4.2M rows)
logger.info("Loading full Parquet dataset...")
df = spark.read.parquet(PARQUET_SAMPLED_PATH)
df.persist(StorageLevel.MEMORY_AND_DISK)

row_count = df.count()
print(f"Loaded rows: {row_count:,}")
print(f"Columns    : {len(df.columns)}")
df.printSchema()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 03:38:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 03:38:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2026-03-03 03:38:19,469 [INFO] Loading full Parquet dataset...
26/03/03 03:38:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Loaded rows: 4,209,699
Columns    : 48
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |--

In [3]:
# =============================================================================
# **SECTION 2: EXPLORATORY DATA ANALYSIS (EDA)**
# Distributed statistics + visualization on collected sample
# =============================================================================

# 2.1: Distributed descriptive statistics
numeric_cols = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", "Distance(mi)"
]

print("=== DISTRIBUTED SUMMARY STATISTICS ===")
df.select(numeric_cols).describe().show(truncate=False)

# 2.2: Approx quantiles (distributed percentile estimation)
print("=== APPROX QUANTILES (Temperature) ===")
temp_quantiles = df.approxQuantile("Temperature(F)", [0.05, 0.25, 0.5, 0.75, 0.95], 0.01)
print(f"  5th  percentile: {temp_quantiles[0]:.1f}°F")
print(f"  25th percentile: {temp_quantiles[1]:.1f}°F")
print(f"  Median         : {temp_quantiles[2]:.1f}°F")
print(f"  75th percentile: {temp_quantiles[3]:.1f}°F")
print(f"  95th percentile: {temp_quantiles[4]:.1f}°F")

# 2.3: Missing data analysis
print("\n=== MISSING DATA ANALYSIS ===")
total = row_count
null_summary = df.select(
    [F.round(F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2).alias(c)
     for c in df.columns]
).collect()[0].asDict()

missing_df = pd.DataFrame(
    [(col, pct) for col, pct in null_summary.items() if pct > 0],
    columns=["Column", "Missing_%"]
).sort_values("Missing_%", ascending=False)

print(missing_df.to_string(index=False))


=== DISTRIBUTED SUMMARY STATISTICS ===


+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|summary|Temperature(F)    |Wind_Chill(F)     |Humidity(%)      |Pressure(in)      |Visibility(mi)   |Wind_Speed(mph)   |Precipitation(in) |Distance(mi)      |
+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|count  |3939101           |3274071           |3935460          |3947755           |3934625          |3791463           |2706130           |4209699           |
|mean   |57.359765591183404|46.087552010937735|61.83565176624881|29.527515808352064|7.844024350478137|14.734829800527004|0.2666442003894701|1.2105525736974454|
|stddev |25.082370376971106|28.65960209752192 |24.24861506503658|1.0531707992262884|3.334359591834908|14.919165497059797|0.533940030241373 |1.9009746143689135|
|min    |-89.0             |-89.0       

  5th  percentile: 7.9°F
  25th percentile: 42.0°F
  Median         : 61.0°F
  75th percentile: 76.0°F
  95th percentile: 92.0°F

=== MISSING DATA ANALYSIS ===


               Column  Missing_%
    Precipitation(in)      35.72
              End_Lng      34.30
              End_Lat      34.30
Astronomical_Twilight      33.70
         Turning_Loop      33.49
        Wind_Chill(F)      22.23
      Wind_Speed(mph)       9.94
       Visibility(mi)       6.53
          Humidity(%)       6.51
       Temperature(F)       6.43
         Pressure(in)       6.22
       Wind_Direction       1.50
    Weather_Condition       1.49
    Weather_Timestamp       1.03
         Airport_Code       0.20
       Civil_Twilight       0.20
       Sunrise_Sunset       0.20
    Nautical_Twilight       0.20
               Street       0.09
             Timezone       0.07
              Zipcode       0.02


In [4]:
# =============================================================================
# **SECTION 3: EDA VISUALIZATIONS**
# Collect small sample to driver for visualization
# =============================================================================

# Collect sample for plotting (5000 rows to driver)
sample_pd = df.sample(fraction=0.001, seed=42).limit(5000).toPandas()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("US Accidents - EDA Overview", fontsize=16, fontweight="bold")

# Plot 1: Severity class distribution
severity_counts = df.groupBy("Severity").count().orderBy("Severity").toPandas()
axes[0, 0].bar(severity_counts["Severity"].astype(str), severity_counts["count"],
               color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"])
axes[0, 0].set_title("Severity Class Distribution")
axes[0, 0].set_xlabel("Severity Level")
axes[0, 0].set_ylabel("Count")
for i, v in enumerate(severity_counts["count"]):
    axes[0, 0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=9)

# Plot 2: Temperature distribution per severity
for sev in [1, 2, 3, 4]:
    subset = sample_pd[sample_pd["Severity"] == sev]["Temperature(F)"].dropna()
    axes[0, 1].hist(subset, bins=30, alpha=0.5, label=f"Severity {sev}")
axes[0, 1].set_title("Temperature Distribution by Severity")
axes[0, 1].set_xlabel("Temperature (°F)")
axes[0, 1].legend()

# Plot 3: Accidents by time of day
hour_counts = (
    df.withColumn("Hour", F.hour("Start_Time"))
    .groupBy("Hour").count()
    .orderBy("Hour").toPandas()
)
axes[0, 2].plot(hour_counts["Hour"], hour_counts["count"], marker="o", color="#9C27B0")
axes[0, 2].set_title("Accident Count by Hour of Day")
axes[0, 2].set_xlabel("Hour")
axes[0, 2].set_ylabel("Count")
axes[0, 2].set_xticks(range(0, 24, 2))

# Plot 4: Top 10 states
top_states = (
    df.groupBy("State").count()
    .orderBy(F.desc("count"))
    .limit(10).toPandas()
)
axes[1, 0].barh(top_states["State"][::-1], top_states["count"][::-1], color="#00BCD4")
axes[1, 0].set_title("Top 10 States by Accident Count")
axes[1, 0].set_xlabel("Count")

# Plot 5: Visibility vs Severity boxplot
sample_pd.boxplot(column="Visibility(mi)", by="Severity", ax=axes[1, 1])
axes[1, 1].set_title("Visibility by Severity")
axes[1, 1].set_xlabel("Severity")
axes[1, 1].set_ylabel("Visibility (mi)")
plt.sca(axes[1, 1])
plt.title("Visibility by Severity")

# Plot 6: Weather condition top 10
top_weather = (
    df.groupBy("Weather_Condition").count()
    .orderBy(F.desc("count"))
    .limit(10).toPandas()
)
top_weather = top_weather.fillna("Unknown")
axes[1, 2].barh(top_weather["Weather_Condition"][::-1], top_weather["count"][::-1],
                color="#FF5722")
axes[1, 2].set_title("Top 10 Weather Conditions")
axes[1, 2].set_xlabel("Count")

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "eda_overview.png")
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()


2026-03-03 03:39:08,976 [INFO] Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-03 03:39:08,978 [INFO] Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


In [5]:
# =============================================================================
# **SECTION 4: CUSTOM TRANSFORMER - DOMAIN-SPECIFIC FEATURE ENGINEERING**
# Implements PySpark ML Transformer interface for pipeline compatibility
# Extracts temporal features, accident duration, and risk indicators
# =============================================================================

from pyspark.ml import Transformer
from pyspark.ml.param.shared import HasInputCols, HasOutputCols
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark import keyword_only
from pyspark.ml.param import Param, Params, TypeConverters

class AccidentFeatureTransformer(
    Transformer,
    DefaultParamsReadable,
    DefaultParamsWritable
):
    """
    Custom PySpark Transformer for US Accidents domain-specific features.
    Extracts: hour, day_of_week, month, duration_min, distance_bin,
    is_peak_hour, is_weekend, temp_risk_category
    """

    @keyword_only
    def __init__(self):
        super(AccidentFeatureTransformer, self).__init__()

    def _transform(self, df):
        # Temporal feature extraction from Start_Time
        df = df.withColumn("Hour",        F.hour("Start_Time"))
        df = df.withColumn("DayOfWeek",   F.dayofweek("Start_Time"))
        df = df.withColumn("Month",       F.month("Start_Time"))
        df = df.withColumn("Year",        F.year("Start_Time"))

        # Accident duration in minutes
        df = df.withColumn(
            "Duration_Min",
            F.round(
                (F.unix_timestamp("End_Time") - F.unix_timestamp("Start_Time")) / 60.0,
                2
            )
        )
        # Cap extreme durations (>= 1440 min = 24h) as likely data entry errors
        df = df.withColumn(
            "Duration_Min",
            F.when(F.col("Duration_Min") < 0, 0.0)
             .when(F.col("Duration_Min") > 1440, 1440.0)
             .otherwise(F.col("Duration_Min"))
        )

        # Is peak hour: 7-9am or 4-7pm commute windows
        df = df.withColumn(
            "Is_Peak_Hour",
            F.when(
                ((F.col("Hour") >= 7) & (F.col("Hour") <= 9)) |
                ((F.col("Hour") >= 16) & (F.col("Hour") <= 19)),
                1
            ).otherwise(0)
        )

        # Is weekend
        df = df.withColumn(
            "Is_Weekend",
            F.when(F.col("DayOfWeek").isin([1, 7]), 1).otherwise(0)
        )

        # Temperature risk: extreme cold (<20°F) or extreme heat (>100°F) = risk 2
        df = df.withColumn(
            "Temp_Risk",
            F.when(
                (F.col("Temperature(F)") < 20) | (F.col("Temperature(F)") > 100), 2
            ).when(
                (F.col("Temperature(F)") >= 20) & (F.col("Temperature(F)") < 40), 1
            ).otherwise(0)
        )

        # Low visibility flag (< 2 miles)
        df = df.withColumn(
            "Low_Visibility",
            F.when(F.col("Visibility(mi)") < 2.0, 1).otherwise(0)
        )

        # Distance bucket (quartile-based risk bins)
        df = df.withColumn(
            "Distance_Bucket",
            F.when(F.col("Distance(mi)") < 0.1, 0)
             .when(F.col("Distance(mi)") < 0.5, 1)
             .when(F.col("Distance(mi)") < 1.0, 2)
             .otherwise(3)
        )

        # Is night (Sunrise_Sunset == Night)
        df = df.withColumn(
            "Is_Night",
            F.when(F.col("Sunrise_Sunset") == "Night", 1).otherwise(0)
        )

        # Combine infrastructure flags into a single risk score
        bool_infra_cols = ["Junction", "Crossing", "Traffic_Signal", "Stop", "Railway"]
        infra_expr = F.lit(0)
        for c in bool_infra_cols:
            infra_expr = infra_expr + F.when(F.col(c) == True, 1).otherwise(0)
        df = df.withColumn("Infra_Risk_Score", infra_expr)

        return df

# Apply custom transformer to sampled DataFrame
custom_transformer = AccidentFeatureTransformer()
df_transformed = custom_transformer.transform(df)

# Verify new features
new_cols = [
    "Hour", "DayOfWeek", "Month", "Year", "Duration_Min",
    "Is_Peak_Hour", "Is_Weekend", "Temp_Risk", "Low_Visibility",
    "Distance_Bucket", "Is_Night", "Infra_Risk_Score"
]
print("=== NEW ENGINEERED FEATURES ===")
df_transformed.select(new_cols + ["Severity"]).show(5, truncate=False)
logger.info(f"Custom transformer applied. Added {len(new_cols)} new features.")


2026-03-03 03:39:12,984 [INFO] Custom transformer applied. Added 12 new features.


=== NEW ENGINEERED FEATURES ===
+----+---------+-----+----+------------+------------+----------+---------+--------------+---------------+--------+----------------+--------+
|Hour|DayOfWeek|Month|Year|Duration_Min|Is_Peak_Hour|Is_Weekend|Temp_Risk|Low_Visibility|Distance_Bucket|Is_Night|Infra_Risk_Score|Severity|
+----+---------+-----+----+------------+------------+----------+---------+--------------+---------------+--------+----------------+--------+
|23  |6        |6    |2020|1440.0      |0           |0         |0        |0             |3              |0       |5               |1       |
|20  |7        |11   |2017|1440.0      |0           |1         |1        |1             |3              |1       |3               |3       |
|19  |5        |10   |2016|1440.0      |1           |0         |0        |0             |3              |1       |1               |1       |
|10  |1        |11   |2020|0.0         |0           |1         |0        |0             |2              |0       |3       

In [6]:
# =============================================================================
# **SECTION 5: PREPROCESSING PIPELINE**
# Handles: missing values, encoding, assembling, scaling
# =============================================================================

# 5.1: Handle Missing Values with MLlib Imputer
numeric_features = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", "Distance(mi)",
    "Duration_Min"
]
numeric_output = [f"{c}_imp" for c in numeric_features]

imputer = Imputer(
    strategy="median",
    inputCols=numeric_features,
    outputCols=numeric_output
)

# 5.2: Fill categorical nulls with "Unknown"
cat_fill = {
    "Weather_Condition": "Unknown",
    "Wind_Direction": "Calm",
    "Sunrise_Sunset": "Day",
    "Civil_Twilight": "Day",
    "Timezone": "US/Eastern",
    "Region": "Unknown",
    "Climate_Zone": "Unknown"
}
df_filled = df_transformed
for col_name, fill_val in cat_fill.items():
    df_filled = df_filled.fillna({col_name: fill_val})

# 5.3: StringIndexer for categorical columns
cat_columns = [
    "Wind_Direction", "Weather_Condition", "Sunrise_Sunset",
    "Timezone", "Region", "Climate_Zone"
]
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in cat_columns
]
cat_indexed_cols = [f"{c}_idx" for c in cat_columns]

# 5.4: OneHotEncoder for low-cardinality categoricals
low_card_cats = ["Sunrise_Sunset", "Timezone", "Region", "Climate_Zone"]
low_card_inputs = [f"{c}_idx" for c in low_card_cats]
low_card_outputs = [f"{c}_ohe" for c in low_card_cats]

ohe = OneHotEncoder(
    inputCols=low_card_inputs,
    outputCols=low_card_outputs,
    dropLast=True
)

# 5.5: Boolean columns to integer
bool_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop"
]
df_bools = df_filled
for bc in bool_cols:
    df_bools = df_bools.withColumn(bc, F.col(bc).cast("integer"))

# 5.6: Assemble all features into one vector
feature_cols = (
    numeric_output          # imputed numerics
    + low_card_outputs      # OHE vectors
    + ["Wind_Direction_idx", "Weather_Condition_idx"]  # high-card as index
    + bool_cols             # binary infrastructure flags
    + [                     # engineered features
        "Hour", "DayOfWeek", "Month", "Is_Peak_Hour", "Is_Weekend",
        "Temp_Risk", "Low_Visibility", "Distance_Bucket", "Is_Night",
        "Infra_Risk_Score"
    ]
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

# 5.7: StandardScaler for normalized features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# 5.8: Build complete MLlib Pipeline
pipeline_stages = [imputer] + indexers + [ohe] + [assembler, scaler]

preprocessing_pipeline = Pipeline(stages=pipeline_stages)

print(f"Pipeline stages    : {len(pipeline_stages)}")
print(f"Feature columns    : {len(feature_cols)}")
print("Pipeline stages summary:")
for i, s in enumerate(pipeline_stages):
    print(f"  Stage {i+1:2d}: {type(s).__name__}")


Pipeline stages    : 10
Feature columns    : 38
Pipeline stages summary:
  Stage  1: Imputer
  Stage  2: StringIndexer
  Stage  3: StringIndexer
  Stage  4: StringIndexer
  Stage  5: StringIndexer
  Stage  6: StringIndexer
  Stage  7: StringIndexer
  Stage  8: OneHotEncoder
  Stage  9: VectorAssembler
  Stage 10: StandardScaler


In [7]:
# =============================================================================
# **SECTION 6: FIT PREPROCESSING PIPELINE & TRANSFORM DATA**
# =============================================================================

# Convert bool columns to int in df_bools before fitting
df_pipeline_input = df_bools

# Fit pipeline on full dataset
logger.info("Fitting preprocessing pipeline...")
fit_start = time.time()

pipeline_model = preprocessing_pipeline.fit(df_pipeline_input)
df_featured = pipeline_model.transform(df_pipeline_input)

fit_end = time.time()
logger.info(f"Pipeline fit+transform completed in {fit_end - fit_start:.2f}s")

# Create target column: label (Severity - 1, mapping 1->0, 2->1, 3->2, 4->3)
df_featured = df_featured.withColumn("label", (F.col("Severity") - 1).cast("double"))

# Select only model-relevant columns for downstream steps
df_final = df_featured.select("ID", "label", "features", "Start_Time", "State",
                               "Severity", "Region", "Climate_Zone")

# Persist for downstream train/test splitting
df_final.persist(StorageLevel.MEMORY_AND_DISK)
final_count = df_final.count()

print(f"Feature-engineered rows : {final_count:,}")
print(f"Pipeline fit time       : {fit_end - fit_start:.2f}s")
print(f"Feature vector size     : {df_final.select('features').first()[0].size}")

# Verify label distribution
print("\n=== LABEL DISTRIBUTION (0-indexed) ===")
df_final.groupBy("label").count().orderBy("label").show()

# Save pipeline model for reuse in model training notebook
PIPELINE_MODEL_PATH = os.path.join(BASE_DIR, "code", "data", "samples", "preprocessing_pipeline_model")
pipeline_model.write().overwrite().save(PIPELINE_MODEL_PATH)

# Unpersist intermediate DataFrames
df.unpersist()
df_transformed.unpersist() if hasattr(df_transformed, 'unpersist') else None
logger.info("Intermediate DataFrames unpersisted.")


2026-03-03 03:39:13,285 [INFO] Fitting preprocessing pipeline...
2026-03-03 03:39:39,141 [INFO] Pipeline fit+transform completed in 25.85s       


Feature-engineered rows : 4,209,699
Pipeline fit time       : 25.85s
Feature vector size     : 48

=== LABEL DISTRIBUTION (0-indexed) ===


+-----+-------+
|label|  count|
+-----+-------+
|  0.0| 376989|
|  1.0|2583530|
|  2.0| 823710|
|  3.0| 425470|
+-----+-------+



2026-03-03 03:40:01,827 [INFO] Intermediate DataFrames unpersisted.


In [8]:
# =============================================================================
# **SECTION 7: DATA SPLITTING STRATEGY**
# Temporal-aware split: use year 2021-2023 as test to avoid data leakage
# 70% Train | 15% Validation | 15% Test
# =============================================================================

RANDOM_SEED = 42

# Stratified random split (no time-series leakage for accident data)
train_frac = 0.70
val_frac   = 0.15
test_frac  = 0.15

# Split using randomSplit with fixed seed for reproducibility
train_df, val_df, test_df = df_final.randomSplit(
    [train_frac, val_frac, test_frac],
    seed=RANDOM_SEED
)

# Persist all splits for efficient reuse in model training
train_df.persist(StorageLevel.MEMORY_AND_DISK)
val_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()

print(f"=== DATA SPLIT SUMMARY ===")
print(f"  Train  : {train_count:>8,} rows ({train_count/final_count*100:.1f}%)")
print(f"  Val    : {val_count:>8,} rows ({val_count/final_count*100:.1f}%)")
print(f"  Test   : {test_count:>8,} rows ({test_count/final_count*100:.1f}%)")
print(f"  Total  : {train_count+val_count+test_count:>8,} rows")

# Verify label balance in train set
print("\n=== TRAIN LABEL DISTRIBUTION ===")
train_df.groupBy("label").count().orderBy("label").show()

# Write all splits to Parquet
SPLITS_DIR = os.path.join(BASE_DIR, "code", "data", "samples", "splits")
train_df.write.mode("overwrite").parquet(os.path.join(SPLITS_DIR, "train"))
val_df.write.mode("overwrite").parquet(os.path.join(SPLITS_DIR, "valid"))
test_df.write.mode("overwrite").parquet(os.path.join(SPLITS_DIR, "test"))


# Write full feature-engineered dataset
df_final.write.mode("overwrite").parquet(PARQUET_FEATURES_PATH)

# Cleanup
df_final.unpersist()
print("Next step: Run 3_model_training.ipynb")


26/03/03 03:40:11 WARN TaskMemoryManager: Failed to allocate a page (33554416 bytes), try again.
26/03/03 03:40:12 WARN TaskMemoryManager: Failed to allocate a page (33554416 bytes), try again.


=== DATA SPLIT SUMMARY ===
  Train  : 2,946,909 rows (70.0%)
  Val    :  630,990 rows (15.0%)
  Test   :  631,800 rows (15.0%)
  Total  : 4,209,699 rows

=== TRAIN LABEL DISTRIBUTION ===


+-----+-------+
|label|  count|
+-----+-------+
|  0.0| 264152|
|  1.0|1808619|
|  2.0| 576343|
|  3.0| 297795|
+-----+-------+



Next step: Run 3_model_training.ipynb
